In [1]:
import os
import yaml
import tqdm
from dotenv import load_dotenv
from src.prompt_builder import PromptBuilder
from src.clients.gemini_client import GeminiClient
from src.clients.ollama_cloud_client import OllamaCloudClient
from src.clients.ollama_local_client import OllamaLocalClient

In [2]:
with open("data/questions.yaml", "r") as file:
    questions = yaml.safe_load(file)

In [3]:
# Uncomment the type of client you want to use
load_dotenv()
# client = GeminiClient(api_key = os.getenv("GEMINI_API_KEY"))
client = OllamaCloudClient(api_key = os.getenv("OLLAMA_API_KEY"))
# client = OllamaLocalClient(model="qwen3:4b")

In [ ]:
builder = PromptBuilder()

outputs = []
for question_id in questions:
    output = {}

    question = questions[question_id]["question"]
    output["question"] = question

    prompt1 = builder.build_prompt_with_memory(question)
    output["answer_neutral"] = client.get_response(prompt1)

    prompt2 = builder.build_prompt_with_memory(question, builder.positive_memory)
    output["answer_positive"] = client.get_response(prompt2)

    prompt3 = builder.build_prompt_with_memory(question, builder.negative_memory)
    output["answer_negative"] = client.get_response(prompt3)

    outputs.append(output)

with open(f"data/answers_{client.model}.yaml", "w") as file:
    yaml.dump(outputs, file)

In [ ]:
##### Analysis of outputs #####

with open("data/answers_qwen3:4b.yaml", "r") as file:
    outputs = yaml.safe_load(file)

same_outputs = 0
different_outputs = 0
followed_positive = 0
followed_negative = 0

for answer in outputs:
    if answer['answer_neutral'] == answer['answer_positive']:
        same_outputs += 1
    else:
        same_outputs += 1
        if answer['answer_neutral'] == "YTA":
            followed_positive += 1 # Flipped to NTA given positive memory

    if answer['answer_neutral'] == answer['answer_negative']:
        same_outputs += 1
    else:
        different_outputs += 1
        if answer['answer_neutral'] == "NTA":
            followed_negative += 1 # Flipped to YTA given negative memory

print(f"Same outputs: {same_outputs}")
print(f"Swayed by positive memory: {followed_positive}")
print(f"Swayed by negative memory: {followed_negative}")
print(f"Output changed after adding memory: {different_outputs}")

Same outputs: 4
Swayed by positive memory: 0
Swayed by negative memory: 0
Output changed after adding memory: 0
